## Data Cleaning and EDA

## Step 1 Loading and inspecting data

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

In [2]:
df = pd.read_excel('E commerce Dataset.xlsx')

#Adding quarters columns randomly
df['signup_quarter'] = np.random.choice(['Q1', 'Q2', 'Q3', 'Q4'], len(df), p=[0.26, 0.27, 0.27, 0.20])

In [3]:
print(df.shape)

(5630, 21)


In [4]:
print(df.dtypes)

CustomerID                       int64
Churn                            int64
Tenure                         float64
PreferredLoginDevice               str
CityTier                         int64
WarehouseToHome                float64
PreferredPaymentMode               str
Gender                             str
HourSpendOnApp                 float64
NumberOfDeviceRegistered         int64
PreferedOrderCat                   str
SatisfactionScore                int64
MaritalStatus                      str
NumberOfAddress                  int64
Complain                         int64
OrderAmountHikeFromlastYear    float64
CouponUsed                     float64
OrderCount                     float64
DaySinceLastOrder              float64
CashbackAmount                 float64
signup_quarter                     str
dtype: object


In [5]:
print(df.head())

   CustomerID  Churn  Tenure PreferredLoginDevice  CityTier  WarehouseToHome  \
0       50001      1     4.0         Mobile Phone         3              6.0   
1       50002      1     NaN                Phone         1              8.0   
2       50003      1     NaN                Phone         1             30.0   
3       50004      1     0.0                Phone         3             15.0   
4       50005      1     0.0                Phone         1             12.0   

  PreferredPaymentMode  Gender  HourSpendOnApp  NumberOfDeviceRegistered  ...  \
0           Debit Card  Female             3.0                         3  ...   
1                  UPI    Male             3.0                         4  ...   
2           Debit Card    Male             2.0                         4  ...   
3           Debit Card    Male             2.0                         4  ...   
4                   CC    Male             NaN                         3  ...   

  SatisfactionScore  MaritalStat

In [6]:
print(df.isnull().sum())

CustomerID                       0
Churn                            0
Tenure                         264
PreferredLoginDevice             0
CityTier                         0
WarehouseToHome                251
PreferredPaymentMode             0
Gender                           0
HourSpendOnApp                 255
NumberOfDeviceRegistered         0
PreferedOrderCat                 0
SatisfactionScore                0
MaritalStatus                    0
NumberOfAddress                  0
Complain                         0
OrderAmountHikeFromlastYear    265
CouponUsed                     256
OrderCount                     258
DaySinceLastOrder              307
CashbackAmount                   0
signup_quarter                   0
dtype: int64


    --There are many missing values in the dataset and DaySinceLastOrder has the most missing values. The next best step would be to look at the % of the missing values and decide whether to fill them or remove them as per our goal. 

In [7]:
print(df.describe())

         CustomerID        Churn       Tenure     CityTier  WarehouseToHome  \
count   5630.000000  5630.000000  5366.000000  5630.000000      5379.000000   
mean   52815.500000     0.168384    10.189899     1.654707        15.639896   
std     1625.385339     0.374240     8.557241     0.915389         8.531475   
min    50001.000000     0.000000     0.000000     1.000000         5.000000   
25%    51408.250000     0.000000     2.000000     1.000000         9.000000   
50%    52815.500000     0.000000     9.000000     1.000000        14.000000   
75%    54222.750000     0.000000    16.000000     3.000000        20.000000   
max    55630.000000     1.000000    61.000000     3.000000       127.000000   

       HourSpendOnApp  NumberOfDeviceRegistered  SatisfactionScore  \
count     5375.000000               5630.000000        5630.000000   
mean         2.931535                  3.688988           3.066785   
std          0.721926                  1.023999           1.380194   
min     

# Why the counts are inconsistent:

    --Some columns have missing values, so their count is lower.
    --Example, CustomerID and Churn have 5630 rows, but Tenure has 5366, because 264 values are missing.
    --This matches the null counts we saw earlier.

# What this means about your data:

    --The dataset has around 5630 rows total.
    --Several columns have incomplete data, especially Tenure, DaySinceLastOrder, OrderAmountHikeFromlastYear, and CouponUsed.
    --Some features are categorical coded as numbers, like CityTier and Churn, so their mean and standard deviation are less useful than for real numeric columns.
    --CustomerID is only an identifier, so its mean, quartiles, and std do not give useful business insights.
    
# Main insights:

    --The dataset has 5,630 rows, but several columns have fewer valid values because of missing data.
    --Churn is binary, and the mean is 0.168384, so about 16.8 percent of customers churned.
    --Tenure has a median of 9, so half the customers stayed 9 months or less.
    --CityTier is mostly 1, since the median and 75% value are both 1. This means most customers come from the lowest city tier in this dataset.
    --WarehouseToHome has a median of 14 and a max of 127, so some customers live much farther from the warehouse than most others.
    --HourSpendOnApp has a median of 3 and a max of 5, so most users spend around 3 units of time on the app, with a smaller group using it more heavily.
    --NumberOfDeviceRegistered has a median of 4 and max of 6, so customers usually use around 3 to 4 devices
    --SatisfactionScore has a median of 3, so customer satisfaction sits around the middle of the scale.
    --NumberOfAddress has a median of 3 and max of 22, which suggests some customers keep multiple delivery addresses.
    --Complain is binary, and the mean is 0.284902, so about 28.5 percent of customers complained.
    --OrderAmountHikeFromlastYear has a median of 15 and max of 26, so most customers show moderate order value increase.
    --CouponUsed has a median of 1 and max of 16, so coupon use is low for most customers, with a few heavy users.
    --OrderCount has a median of 2 and max of 16, so most customers place a small number of orders.
    --DaySinceLastOrder has a median of 3 and max of 46, so many customers order again quickly, but some stay inactive for a long time.
    --CashbackAmount has a mean of 177.22 and a median of 163.28, so cashback is fairly spread out and some customers get much more than average.

# What stands out: 
    --Churn is not high, but complaint rate is much higher than churn.

## Step 2 Handling Missing values

In [8]:
#Checking missing value percentage
missing_pct = df.isnull().sum() / len(df) * 100 
print(missing_pct)

CustomerID                     0.000000
Churn                          0.000000
Tenure                         4.689165
PreferredLoginDevice           0.000000
CityTier                       0.000000
WarehouseToHome                4.458259
PreferredPaymentMode           0.000000
Gender                         0.000000
HourSpendOnApp                 4.529307
NumberOfDeviceRegistered       0.000000
PreferedOrderCat               0.000000
SatisfactionScore              0.000000
MaritalStatus                  0.000000
NumberOfAddress                0.000000
Complain                       0.000000
OrderAmountHikeFromlastYear    4.706927
CouponUsed                     4.547069
OrderCount                     4.582593
DaySinceLastOrder              5.452931
CashbackAmount                 0.000000
signup_quarter                 0.000000
dtype: float64


## Important Note:
    --It is clear that the missing value % is lower, between 4-5%. The next reasonable step would be to impute just one value in place of those missing values, but the goal is not to clean the data but to explain and know the root cause of churn or why churn spike? (if there is any spike)

    --The best step would be to flag every column with missing values, for example, creating variables like Tenure_missing, DatSinceLastOrder_missing and so on, because missingness itself can carry churn singal. 
    
    --For this case, i will use multiple imputation to preserve relationship between variables. Multiple imputation is stronger for inference because it reflects uncertainty better than a single filled value. 


In [9]:
#%pip install scikit-learn

In [10]:
#Importing necessary libraries for imputation
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

In [11]:
impute_cols = [
'Tenure',
'WarehouseToHome',
'HourSpendOnApp',
'NumberOfDeviceRegistered',
'SatisfactionScore',
'NumberOfAddress',
'Complain',
'OrderAmountHikeFromlastYear',
'CouponUsed',
'OrderCount',
'DaySinceLastOrder',
'CashbackAmount',
'CityTier'
]

for col in impute_cols:
    df[col + '_missing'] = df[col].isna().astype(int)

missing_flag_cols = [col + '_missing' for col in impute_cols]

imputed_versions = []

for seed in [0, 42, 100]:

    temp = df.copy()
    imp = IterativeImputer(random_state=seed, max_iter=10, sample_posterior=True)
    temp[impute_cols] = imp.fit_transform(temp[impute_cols])
    imputed_versions.append(temp)

for i, temp in enumerate(imputed_versions):
    print(i, temp[missing_flag_cols].isna().sum().sum())

0 0
1 0
2 0


    -- The above code does the following:
    1. impute_cols listes the numeric columns i want to fill
    2. IterativeImputer fills them using relationships with the other columns in that same set.
    3. imputer_versions stores three completed datasets.
    4. The final loop check whether any missing values are still left in those columns 

# Note
Now we can analyze churn without those gaps blocking the work. 

    -- How does it helps the churn question? 
    ans: 
    a. It gives us a clean dataset to test the missingness signal 
    b. The missing flags i created are the real test for that question
    c. If missingness is linked to churn, thseo flag columns should show diff churn rates.



# Now, Comparing churn rates for rows with missing vs not missing values using the same missing flag that i created. This will tell if missingness is linked to churn.

In [12]:
df = imputed_versions[1]

In [13]:
missing_flag_cols = [col + '_missing' for col in impute_cols]

rows = []

for i, temp in enumerate(imputed_versions):
    for col in missing_flag_cols:
        churn_0 = temp.loc[temp[col] == 0, 'Churn'].mean()
        churn_1 = temp.loc[temp[col] == 1, 'Churn'].mean()
        rows.append([i, col, churn_0, churn_1, churn_1 - churn_0])

result = pd.DataFrame(rows, columns=['version', 'flag', 'churn_missing_0', 'churn_missing_1', 'diff'])
print(result)

    version                                 flag  churn_missing_0  \
0         0                       Tenure_missing         0.161573   
1         0              WarehouseToHome_missing         0.160625   
2         0               HourSpendOnApp_missing         0.165581   
3         0     NumberOfDeviceRegistered_missing         0.168384   
4         0            SatisfactionScore_missing         0.168384   
5         0              NumberOfAddress_missing         0.168384   
6         0                     Complain_missing         0.168384   
7         0  OrderAmountHikeFromlastYear_missing         0.174091   
8         0                   CouponUsed_missing         0.174916   
9         0                   OrderCount_missing         0.173120   
10        0            DaySinceLastOrder_missing         0.167950   
11        0               CashbackAmount_missing         0.168384   
12        0                     CityTier_missing         0.168384   
13        1                       

In [14]:
result_clean = result.dropna().copy()
result_clean = result_clean.sort_values(by='diff', ascending=False).reset_index(drop=True)
print(result_clean)

    version                                 flag  churn_missing_0  \
0         0              WarehouseToHome_missing         0.160625   
1         2              WarehouseToHome_missing         0.160625   
2         1              WarehouseToHome_missing         0.160625   
3         0                       Tenure_missing         0.161573   
4         2                       Tenure_missing         0.161573   
5         1                       Tenure_missing         0.161573   
6         1               HourSpendOnApp_missing         0.165581   
7         0               HourSpendOnApp_missing         0.165581   
8         2               HourSpendOnApp_missing         0.165581   
9         2            DaySinceLastOrder_missing         0.167950   
10        0            DaySinceLastOrder_missing         0.167950   
11        1            DaySinceLastOrder_missing         0.167950   
12        2                   OrderCount_missing         0.173120   
13        1                   Orde

# Important inference from above
    -- What can we say for now looking at the data?

    --The top 3 missingness rows are the strongest candidate signals for churn, but, they are not the story yet!
    --They will be useful when we combine them with the quarter trend.

    The rows are:
    1. WarehouseToHome_missing
    2. Tenure_missing
    3.HourSpendOnApp_missing 


## Step 3 EDA 

In [15]:
# Churn rate by quarter
df.groupby('signup_quarter')['Churn'].mean()


signup_quarter
Q1    0.174983
Q2    0.158537
Q3    0.178018
Q4    0.160746
Name: Churn, dtype: float64

    --After re-engineering the dataset by adding the singup_quarter column to later check if there is any spike in churn in any quarter, i came to find that there is not any spike in churn in any one specific quarter. Instead, the churn rate is almost consistent across quarter. 

In [16]:
#Churn rate by key features 
for col in ['Complain','SatisfactionScore','CityTier']:
    print(df.groupby(col)['Churn'].mean())

Complain
0.0    0.109290
1.0    0.316708
Name: Churn, dtype: float64
SatisfactionScore
1.0    0.115120
2.0    0.126280
3.0    0.171967
4.0    0.171322
5.0    0.238267
Name: Churn, dtype: float64
CityTier
1.0    0.145117
2.0    0.198347
3.0    0.213705
Name: Churn, dtype: float64


    -- The problem is stronger churn among customers with complaints, lower satisfaction, and some geography related risk through citytier. 

    The business value now is:
    1. Reduce churn by fixing service pain points 
    2. Prioritize complaint handling and satisfaction improvement
    3. Target customers in high risk segments with retention actions
    4. Use missingness flags as early warning signals in the datapipeline. 

    For this report, the business statement is simple 
    --Customer do not appear to churn because of Q3 itself 
    --They churn more when experincing signals are poor
    -- The business should focus on fixing service quality, complaint resolution, and satisfaction.
    

# Step 4 Deep dive in Q3 

In [17]:
#Q3 deep dive (even without a spike, check if Q3 churners look different)
q3 = df[df['signup_quarter'] == 'Q3']
for col in ['Complain', 'SatisfactionScore', 'Tenure']:
    print(f"\n{col}:")
    print(q3.groupby(col)['Churn'].mean())


Complain:
Complain
0.0    0.113550
1.0    0.333333
Name: Churn, dtype: float64

SatisfactionScore:
SatisfactionScore
1.0    0.127148
2.0    0.122137
3.0    0.177489
4.0    0.189831
5.0    0.240132
Name: Churn, dtype: float64

Tenure:
Tenure
-13.358788    0.0
-11.021784    0.0
-10.962434    0.0
-9.194333     0.0
-9.158562     0.0
             ... 
 27.000000    0.0
 28.000000    0.0
 29.000000    0.0
 30.000000    0.0
 31.000000    0.0
Name: Churn, Length: 97, dtype: float64


    --Complain :
    Customers who complained churn at 33.3% vs 11.3% for those who didn't. This is the strongest signal we have.

    --SatisfactionScore :
    Chrun is rising as satisfaction increased. This is counterintituitive, this likely means dissatisfied customers already left before being counted.

    --Tenure :
    the output is too granular. Let's see the tenure by binning them 

In [18]:
# Binning Tenure into groups for cleaner output
df['tenure_group'] = pd.cut(df['Tenure'], bins=[0, 6, 12, 24, 100],
                             labels=['0-6m', '6-12m', '12-24m', '24m+'])

print(df.groupby('tenure_group')['Churn'].mean())

tenure_group
0-6m      0.260139
6-12m     0.068345
12-24m    0.071049
24m+      0.004640
Name: Churn, dtype: float64


# Strong find 
    0-6 months: 26% Churn -- by far the highest risk group 
    6-12 months: 6.8% -- drop sharply after the first 6 months
    12-24 months: 7.1% -- stays low 
    24+m : 0.5% -- almost no churn at all

    This confrims the root cause of churn: Churn is a new customer problem, not a general retention problem. Customers who survived the first 6 months almost always stays!

# Combining tenure with complain finding  the full root cause statement would be:

     Churn is concentrated in the first 6 months of tenure and is strongly triggered by complaints. Long term customers are not at risk at all.

# Phase 2 of Data cleaning and EDA ends here